# TabFM context-strategy ablations

Time-based split: context/train pool = seasons before 2025, validate on season 2025.
2026 is excluded by `tabfm_model.load_data`'s `HOLDOUT_YEAR` guard and is never touched here -- only after picking a winning strategy on this validation set should it be re-run on 2026, and only once real 2026 rows exist.

In [ ]:
import sys
sys.path.insert(0, "src/scripts")

import numpy as np
import pandas as pd

from tabfm_model import load_data, build_context, _run_tabfm, TARGET
from evaluation import player_metrics, team_results

In [ ]:
VALIDATION_SEASON = 2025
CONTEXT_SIZE = 100

df = load_data()
assert (df["season"] < 2026).all()  # structural 2026 holdout, not just notebook discipline

val = df[df["season"] == VALIDATION_SEASON].copy()
context_pool = df[df["season"] < VALIDATION_SEASON]
print(f"Context pool: {len(context_pool)} rows, seasons {sorted(context_pool['season'].unique())}")
print(f"Validation:   {len(val)} rows, {val['match_id'].nunique()} matches, season {VALIDATION_SEASON}")

In [ ]:
def predict_strategy(val_df: pd.DataFrame, full_df: pd.DataFrame, strategy: str, context_size: int = CONTEXT_SIZE, seed: int = 42, **ratios) -> np.ndarray:
    """Predict every validation row using that row's own real features as the query -- the labels are only used for scoring, never fed to TabFM."""
    preds = np.empty(len(val_df))
    for i, (_, row) in enumerate(val_df.iterrows()):
        context = build_context(row["date"], [row["player"]], strategy, context_size, seed=seed, df=full_df, **ratios)
        preds[i] = _run_tabfm(context, row)
    return preds

In [ ]:
EXPERIMENTS = {
    "Experiment 1: player-only": dict(strategy="player"),
    "Experiment 2: global": dict(strategy="global"),
    "Experiment 3: player + similar": dict(strategy="player_similar"),
    "Experiment 4: hybrid (0.4/0.3/0.3)": dict(strategy="hybrid", player_ratio=0.4, similar_ratio=0.3, global_ratio=0.3),
}

predictions = {}
player_rows = []
for name, cfg in EXPERIMENTS.items():
    print(f"Running {name}...")
    preds = predict_strategy(val, df, **cfg)
    val[f"pred_{name}"] = preds
    predictions[name] = preds
    player_rows.append({"Experiment": name, **player_metrics(val, preds)})

pd.DataFrame(player_rows).set_index("Experiment").round(3)

In [ ]:
# A few hybrid ratio variants, on top of the default 0.4/0.3/0.3 already run above
RATIO_VARIANTS = {
    "hybrid (0.5/0.25/0.25)": dict(player_ratio=0.5, similar_ratio=0.25, global_ratio=0.25),
    "hybrid (0.33/0.33/0.34)": dict(player_ratio=0.33, similar_ratio=0.33, global_ratio=0.34),
}

for variant_name, ratios in RATIO_VARIANTS.items():
    label = f"Experiment 4: {variant_name}"
    print(f"Running {label}...")
    preds = predict_strategy(val, df, strategy="hybrid", **ratios)
    val[f"pred_{label}"] = preds
    predictions[label] = preds
    player_rows.append({"Experiment": label, **player_metrics(val, preds)})

pd.DataFrame(player_rows).set_index("Experiment").round(3)

In [ ]:
# Team-level: predicted XI vs actual vs the Dream Team (ILP on actual points), per experiment
matches = team_results(val, predictions)
summary = matches.groupby("Model", sort=False).agg(**{
    "Actual points of XI": ("Actual Points", "mean"),
    "% of Dream Team": ("Pct of Dream Team", "mean"),
    "Overlap with Dream XI": ("Overlap with Dream XI", "mean"),
    "Team total MAE": ("MAE", "mean"),
})
summary.round(1)

Pick the strategy/ratio with the best validation numbers above. Only then re-run it against 2026 -- and only once real 2026 rows exist in the data (`load_data`'s `HOLDOUT_YEAR` guard makes running it on 2026 today impossible even by accident).